In [1]:
!wget -O data.zip "https://zenodo.org/records/15183245/files/Audio.zip?download=1"

# 2. Extract
!mkdir -p /content/audio_data
!unzip -q data.zip -d /content/audio_data

# 3. Check files
import os
count = len([f for f in os.listdir('/content/audio_data/Audio') if f.endswith('.mp3')])
print(f"congratulations! Total {count} files downloaded.")

--2026-03-07 15:02:17--  https://zenodo.org/records/15183245/files/Audio.zip?download=1
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 137.138.153.219, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 923431009 (881M) [application/octet-stream]
Saving to: ‘data.zip’

data.zip            100%[===================>] 880.65M   898KB/s    in 19m 16s 

2026-03-07 15:21:34 (780 KB/s) - ‘data.zip’ saved [923431009/923431009]

congratulations! Total 8579 files downloaded.


In [ ]:
import os
count = len([f for f in os.listdir('/content/audio_data/Audio') if f.endswith('.mp3')])
print(f"congratulations! Total {count} files downloaded.")

congratulations! Total 8579 files downloaded.


In [3]:
!pip install --upgrade transformers tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 66.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
import os
import csv
import torch
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2Model

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = 'kingabzpro/wav2vec2-large-xls-r-300m-Urdu'
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name).to(device) # Move to GPU

def extract_audio_features(audio_file):
    try:
        # Load audio (16kHz is correct for Wav2Vec2)
        y, sr = librosa.load(audio_file, sr=16000)

        # Audio length control (optional: take first 10 seconds to avoid OOM)
        # y = y[:16000*10]

        inputs = processor(y, return_tensors="pt", padding="longest", sampling_rate=sr).to(device)

        with torch.no_grad():
            model_output = model(**inputs)

        # Mean pooling to get a single vector per audio file
        features = model_output.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return features.tolist()
    except Exception as e:
        print(f"Error processing {audio_file}: {e}")
        return None

def process_and_save(folder_path, output_file):
    # Open CSV once and write row by row to save memory
    with open(output_file, 'w', newline='') as csvfile:
        csv_writer = csv.writer(csvfile)
        header_written = False

        for root, _, files in os.walk(folder_path):
            for file in files:
                if file.endswith(".mp3"):
                    audio_path = os.path.join(root, file)
                    features = extract_audio_features(audio_path)

                    if features:
                        if not header_written:
                            # Create Header dynamically based on feature size
                            csv_writer.writerow(["Link", "Label"] + [f"F_{i}" for i in range(len(features))])
                            header_written = True

                        # Label: file[0] (as per your original logic)
                        csv_writer.writerow([audio_path, file[0]] + features)

audio_directory = "audio_data/Audio"
output_file = "/content/drive/MyDrive/audio_features.csv"

process_and_save(audio_directory, output_file)
print("Processing Complete!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: kingabzpro/wav2vec2-large-xls-r-300m-Urdu
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing Complete!


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Check if CUDA (GPU) is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the data from your CSV file
data = pd.read_csv("/content/drive/MyDrive/audio_features.csv")
Y_AF = data["Label"]
# Split data into features (X) and labels (y)
X_AF = data.drop(columns=["Link", "Label"])


# Use LabelEncoder to convert string labels to numerical labels
label_encoder = LabelEncoder()
Y_AF = label_encoder.fit_transform(Y_AF)

sequence_length = 20
# --- Step 1: Split raw data first ---
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_AF.values, Y_AF, test_size=0.2, random_state=42, shuffle=False
)

def create_sequences(features, labels, seq_len):
    seqs, lbls = [], []
    for i in range(len(features) - seq_len + 1):
        seqs.append(features[i:i+seq_len])
        lbls.append(labels[i + seq_len - 1])
    return torch.tensor(seqs, dtype=torch.float32), torch.tensor(lbls, dtype=torch.long)

# --- Step 2: Create sequences separately ---
X_train_A, y_train_A = create_sequences(X_train_raw, y_train_raw, sequence_length)
X_test_A, y_test_A = create_sequences(X_test_raw, y_test_raw, sequence_length)

# Move to device
X_train_A, y_train_A = X_train_A.to(device), y_train_A.to(device)
X_test_A, y_test_A = X_test_A.to(device), y_test_A.to(device)
# Create a DataLoader for the training set (optional but useful for mini-batch training)
batch_size = 32  # Adjust as needed
train_dataset = TensorDataset(X_train_A, y_train_A)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Define an LSTM-based model
class EmotionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(EmotionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,dropout=dropout if num_layers > 1 else 0,bidirectional=True)
        self.fc = nn.Linear(hidden_size, num_classes)


        self.fc = nn.Linear(hidden_size * 2, num_classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = torch.max(out, dim=1)[0]  # Take the output from the last time step
        out = self.fc(out)
        return out

# Define the LSTM model hyperparameters
input_size = X_train_A.shape[2]  # Input size based on the number of features in each time step
hidden_size = 64
num_layers = 2  # You can adjust this as needed
num_classes = len(label_encoder.classes_)

# Initialize the model and move it to the GPU
model = EmotionLSTM(input_size, hidden_size, num_layers, num_classes).to(device)

# Define a loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}')

# Set the model to evaluation mode
model.eval()

# Make predictions on the test set
with torch.no_grad():
    outputs = model(X_test_A)
    _, predicted = torch.max(outputs, 1)

# Move the predictions to the CPU and convert them to a NumPy array
predicted = predicted.cpu().numpy()

# Calculate accuracy
accuracy = accuracy_score(y_test_A.cpu().numpy(), predicted)
print("Accuracy:", accuracy)

Epoch [10/50], Loss: 1.3697
Epoch [20/50], Loss: 1.2543
Epoch [30/50], Loss: 1.1654
Epoch [40/50], Loss: 1.1062
Epoch [50/50], Loss: 1.0618
Accuracy: 0.5191514437242192
